# Day 083 — Exercise 4: run_plan — End to End

**What you'll build:** `run_plan` — the end-to-end orchestrator that plans a goal, sorts the tasks by dependency, and executes them in order.

**Why it matters:** this is the payoff of the day's three earlier pieces. One function takes a goal, hands it to the LLM for decomposition, sorts the task list, feeds results forward, and returns the final answer — all with a `max_tasks` guard so a runaway plan can't hang the gate.

In [ ]:
import json

_PLAN_JSON = json.dumps([
    {'id': 't1', 'title': 'Gather facts',
     'description': 'Collect the relevant information.', 'depends_on': []},
    {'id': 't2', 'title': 'Draft outline',
     'description': 'Organize the facts into an outline.', 'depends_on': ['t1']},
    {'id': 't3', 'title': 'Write summary',
     'description': 'Write the final summary.', 'depends_on': ['t2']},
])

def _mock_planner(plan_json=None, task_result='Task done.'):
    """Return an llm_fn: the plan JSON on planning calls, task_result on execution calls."""
    plan = plan_json if plan_json is not None else _PLAN_JSON
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'json array' in system.lower() or 'planning' in system.lower():
            return plan
        return task_result
    return _fn

def _mock_executor(task):
    return 'Result: ' + task.title
import json
from dataclasses import dataclass, field

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def safe_parse_list(text):
    """Slice first '[' to last ']' and parse. Returns list|None. Never raises."""
    start, end = text.find("["), text.rfind("]")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, list) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the Task dataclass ────────────────────────────────────────────────────────
@dataclass
class Task:
    """One step in a plan.

    Attributes:
        id:          short snake_case identifier (e.g. 't1', 'write_outline').
        title:       brief human-readable label (5 words max).
        description: one sentence describing what to do.
        depends_on:  ids of tasks that must complete before this one.
        status:      'pending' | 'done' | 'failed'.
        result:      the output of executing this task.
    """
    id: str
    title: str
    description: str
    depends_on: list = field(default_factory=list)
    status: str = "pending"
    result: str = ""

# ── planning: ask the LLM to decompose a goal ─────────────────────────────────
def build_plan_prompt(goal, context=None):
    """Build a prompt that asks the LLM to break a goal into a JSON task list."""
    system = "\n".join([
        "You are a planning assistant. Break the goal into an ordered list of tasks.",
        "",
        "Return ONLY a JSON array. Each item must have these exact keys:",
        '  "id": short snake_case identifier (t1, t2, ...)',
        '  "title": brief label (5 words max)',
        '  "description": one sentence - what to do',
        '  "depends_on": list of task ids that must finish before this one ([] if none)',
        "",
        "Return ONLY the JSON array. No prose, no markdown fences.",
    ])
    user_parts = ["Goal: " + str(goal)]
    if context:
        user_parts.append("Context: " + str(context))
    return [{"role": "system", "content": system},
            {"role": "user", "content": "\n".join(user_parts)}]


def parse_plan(text):
    """Extract a task list from LLM output. Returns list[Task]; never raises.

    Tolerates markdown fences, prose before/after, missing fields, and invalid
    JSON. Invalid or missing fields are filled with safe defaults so any
    parseable item becomes a valid Task.
    """
    items = safe_parse_list(text) or []
    tasks = []
    for i, item in enumerate(items):
        if not isinstance(item, dict):
            continue
        tasks.append(Task(
            id=str(item.get("id", "t" + str(i + 1))),
            title=str(item.get("title", "Task " + str(i + 1))),
            description=str(item.get("description", "")),
            depends_on=[str(d) for d in item.get("depends_on", [])
                        if isinstance(d, str)],
        ))
    return tasks

# ── dependency ordering: Kahn's topological sort ──────────────────────────────
def topo_sort(tasks):
    """Sort tasks so every dependency comes before the task that needs it.

    Uses Kahn's algorithm (BFS on a DAG). If a cycle exists the cyclic tasks
    are appended at the end in their original order rather than raising, so
    execution can still proceed on the non-cyclic portion.
    """
    by_id = {t.id: t for t in tasks}
    # count incoming edges (how many unresolved deps each task has)
    in_deg = {t.id: 0 for t in tasks}
    for t in tasks:
        for dep in t.depends_on:
            if dep in in_deg:
                in_deg[t.id] += 1
    # start with tasks that have no deps
    queue = [t.id for t in tasks if in_deg[t.id] == 0]
    order = []
    while queue:
        tid = queue.pop(0)
        order.append(by_id[tid])
        # for every task that listed tid as a dep, reduce its in-degree
        for t in tasks:
            if tid in t.depends_on:
                in_deg[t.id] -= 1
                if in_deg[t.id] == 0:
                    queue.append(t.id)
    # cycle guard: any task not yet emitted has a circular dependency
    done_ids = {t.id for t in order}
    for t in tasks:
        if t.id not in done_ids:
            order.append(t)
    return order

# ── executing tasks ───────────────────────────────────────────────────────────
def build_execution_context(task, results):
    """Render prior results that this task depends on, for injection into the prompt."""
    lines = ["You are executing one step of a multi-task plan."]
    prior = [(dep, results[dep]) for dep in task.depends_on if dep in results]
    if prior:
        lines.append("Results from earlier steps:")
        for dep_id, res in prior:
            lines.append("  " + dep_id + ": " + str(res))
    lines.append("Task: " + task.title)
    lines.append("Description: " + task.description)
    lines.append("Complete this task concisely.")
    return "\n".join(lines)


def execute_task(task, results, executor_fn=None, llm_fn=None):
    """Run one task. Returns the result string; never raises.

    If executor_fn is provided, call executor_fn(task) -> str.
    Otherwise use the LLM, passing prior results as context.
    Exceptions are caught and returned as 'Error: ...' strings.
    """
    try:
        if executor_fn is not None:
            return str(executor_fn(task))
        context = build_execution_context(task, results)
        messages = [{"role": "system", "content": context},
                    {"role": "user", "content": "Execute this task now."}]
        return call_llm(messages, llm_fn=llm_fn)
    except Exception as exc:
        return "Error: " + str(exc)


## Task

`run_plan(goal, executor_fn=None, llm_fn=None, max_tasks=20) -> dict`

1. Call `call_llm(build_plan_prompt(goal), llm_fn=llm_fn)` to get the plan.
2. `parse_plan` and `topo_sort` it (cap at `max_tasks` before sorting).
3. If empty, return `{'tasks': [], 'results': {}, 'answer': 'No plan generated.'}`.
4. Loop: for each task, `execute_task(task, results, ...)`, update `task.result`, `task.status = 'done'`, and `results[task.id] = result`.
5. Return `{'tasks': ordered, 'results': results, 'answer': last task's result}`.

## Your Implementation

In [ ]:
def run_plan(goal, executor_fn=None, llm_fn=None, max_tasks=20):
    """Plan a goal, sort by dependencies, and execute step by step.
    Returns {'tasks': list[Task], 'results': {id: result}, 'answer': str}.
    """
    raise NotImplementedError


In [ ]:

# ── end-to-end plan runner ────────────────────────────────────────────────────
def run_plan(goal, executor_fn=None, llm_fn=None, max_tasks=20):
    """Plan a goal, sort by dependencies, and execute step by step.

    Returns {"tasks": list[Task], "results": {id: result}, "answer": str}.
    The answer is the result of the last task in execution order.
    max_tasks caps the plan so a model that returns 1000 tasks cannot hang the gate.
    """
    plan_text = call_llm(build_plan_prompt(goal), llm_fn=llm_fn)
    tasks = parse_plan(plan_text)
    if not tasks:
        return {"tasks": [], "results": {}, "answer": "No plan generated."}
    ordered = topo_sort(tasks[:max_tasks])
    results = {}
    for task in ordered:
        result = execute_task(task, results,
                              executor_fn=executor_fn, llm_fn=llm_fn)
        task.result = result
        task.status = "done"
        results[task.id] = result
    answer = ordered[-1].result if ordered else "No tasks executed."
    return {"tasks": ordered, "results": results, "answer": answer}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    r = run_plan('test goal',
                 executor_fn=_mock_executor,
                 llm_fn=_mock_planner())
    assert 'tasks' in r and 'results' in r and 'answer' in r
    score += 1; print("✅ run_plan returns {tasks, results, answer}")

    assert len(r['tasks']) == 3 and r['tasks'][0].status == 'done'
    score += 1; print("✅ all tasks are executed and marked done")

    ids = [t.id for t in r['tasks']]
    assert ids.index('t1') < ids.index('t2') < ids.index('t3')
    score += 1; print("✅ tasks run in dependency order")

    assert r['results']['t2'] == 'Result: Draft outline'
    assert r['answer'] == 'Result: Write summary'
    score += 1; print("✅ results and answer are correct")

    empty = run_plan('x', executor_fn=_mock_executor,
                     llm_fn=_mock_planner(plan_json='no json here'))
    assert empty['tasks'] == [] and 'No plan' in empty['answer']
    score += 1; print("✅ empty plan returns a safe fallback (no crash)")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── end-to-end plan runner ────────────────────────────────────────────────────
def run_plan(goal, executor_fn=None, llm_fn=None, max_tasks=20):
    """Plan a goal, sort by dependencies, and execute step by step.

    Returns {"tasks": list[Task], "results": {id: result}, "answer": str}.
    The answer is the result of the last task in execution order.
    max_tasks caps the plan so a model that returns 1000 tasks cannot hang the gate.
    """
    plan_text = call_llm(build_plan_prompt(goal), llm_fn=llm_fn)
    tasks = parse_plan(plan_text)
    if not tasks:
        return {"tasks": [], "results": {}, "answer": "No plan generated."}
    ordered = topo_sort(tasks[:max_tasks])
    results = {}
    for task in ordered:
        result = execute_task(task, results,
                              executor_fn=executor_fn, llm_fn=llm_fn)
        task.result = result
        task.status = "done"
        results[task.id] = result
    answer = ordered[-1].result if ordered else "No tasks executed."
    return {"tasks": ordered, "results": results, "answer": answer}
```

**Why cap at `max_tasks` before sorting?** The sort and execution loop both iterate over the task list. A model that returns 100 tasks would cause 100 executor calls and a very large sort. Capping first keeps both bounded — same principle as `max_iterations` on the agent loops.

</details>